# M6 · How training works

**Tiny Transformer, episode M6.** All 9,417 of the model's learned numbers started random. Here you run what changed
them: draw a batch of 64 windows from the training text, score it with one number (the loss), give every learned number a
gradient and take one small step; then repeat 500 times and watch the loss on held-back text fall. Last, let the model write at
step 0, at step 500 and at the recorded run's best step, and score it against two baselines.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/M6-training-loop.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook M6-training-loop.ipynb`.

**Computed here:** the batch, every loss, the gradients, 500 training steps from the recorded seed, the held-back checks, what the
model writes and the whole-text scores. **Recorded:** the trained numbers of the best step, 98,000 (`prompter.json`). The
first 500 steps are the recorded run's own: the notebook asserts that its checks equal the run's log (the course replayed them
with `training/prompter/replay.py`, which matched the log at every check).

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, math, os, urllib.request
import numpy as np
import torch

torch.set_num_threads(1)   # one thread keeps the run steady on any machine (the recorded run used 4: it moves only the 7th decimal)
print(torch.__version__)

2.10.0+cu128


**The text, the model and this episode's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`,
copied in whole; then this episode's teaching functions (`code/training_loop.py`).

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
import math

import torch


def get_batch(ids, B, T, g=None):
    """B windows of T + 1 characters from random places in the text: the inputs x, and the targets y, moved one along."""
    starts = torch.randint(len(ids) - T - 1, (B,), generator=g)   # [B]        where each window starts
    x = torch.stack([ids[i:i + T] for i in starts])               # [B×T]      the inputs
    y = torch.stack([ids[i + 1:i + T + 1] for i in starts])       # [B×T]      the targets: the next character at every position
    return x, y


def loss_of(model, x, y):
    """The loss of a batch, step by step: the probability each position gave its right answer, minus its log, averaged."""
    logits, _ = model(x)                                          # [B×T×65]   a score for every character, at every position
    probs = torch.softmax(logits, dim=-1)                         # [B×T×65]   probabilities: each position's 65 add up to 1
    p_right = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)       # [B×T]      the probability of the right next character
    return -p_right.log().mean()                                  # []         minus its natural log, averaged: one number


def lr_at(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """The learning rate of a step: up from 0 during the warm-up, then down a cosine curve to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup                                 # warm-up: 0 up to 0.003 over the first 200 steps
    p = min(1.0, (step - warmup) / max(1, max_steps - warmup))    # how far through the rest of the run, 0 to 1
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * p))   # 0.003 down to 0.0003


def train_step(model, opt, x, y, lr):
    """One training step: the batch's loss, a gradient for every learned number, and a small step against each gradient."""
    for group in opt.param_groups:
        group["lr"] = lr                                          # this step's learning rate
    _, loss = model(x, y)                                         # []         the batch's loss (model.py's, the same as loss_of)
    opt.zero_grad(set_to_none=True)                               # forget the last step's gradients
    loss.backward()                                               # backpropagation: every learned number gets a gradient, .grad
    opt.step()                                                    # AdamW moves every number a small step against its gradient
    return loss.item()


@torch.no_grad()
def average_loss(model, batches):
    """The loss over a list of batches (x, y), every answer counted once: on fixed held-back batches, the check of training."""
    total = sum(model(x, y)[1].item() * y.numel() for x, y in batches)   # each batch's loss × how many answers it holds
    return total / sum(y.numel() for _, y in batches)                     # []  over all the answers: one number

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # [1115394]  the whole text as ids
n = int(0.9 * len(ids))
train_ids, val_ids = ids[:n], ids[n:]                       # the first 90% trains; the last 10% is held back
print('text', tuple(ids.shape), ' training', tuple(train_ids.shape), ' held back', tuple(val_ids.shape))
assert (len(train_ids), len(val_ids)) == (1003854, 111540)

text (1115394,)  training (1003854,)  held back (111540,)


## 1 · A batch

A training step reads a **batch**: 64 windows of 33 characters, each from a random place in the training text. The first 32
characters of each window are the inputs; the same window moved one along gives the targets. The recorded run seeded PyTorch's
random numbers with 1337, built the model (which draws its starting numbers), and then drew its first batch: the same calls,
in the same order, give the same 64 windows. Every shape gains one axis in front, the batch: `[32]` becomes `[64×32]`.

In [7]:
torch.manual_seed(1337)                    # the recorded run's seed
model = Prompter(cfg)                    # its starting numbers: step 0, untrained
x, y = get_batch(train_ids, 64, 32)       # its first batch
print('inputs', tuple(x.shape), ' targets', tuple(y.shape), ' questions', y.numel())
print('first window, inputs: ', repr(tok.decode(x[0])))
print('first window, targets:', repr(tok.decode(y[0])))
assert (tuple(x.shape), tuple(y.shape), y.numel()) == ((64, 32), (64, 32), 2048)
assert torch.equal(x[:, 1:], y[:, :-1])                  # the targets are the inputs, moved one along
assert [tok.decode(w) for w in x[:4]] == ['no doubt; and so shall Clarence ', '\nVery true, and but a month old.', 'ended to thee.\n\nAUTOLYCUS:\nI am ', 'sir, and hanged after.\nProclaim ']   # the recorded first batch

inputs (64, 32)  targets (64, 32)  questions 2048
first window, inputs:  'no doubt; and so shall Clarence '
first window, targets: 'o doubt; and so shall Clarence t'


### ✋ Your turn

Make a smaller batch: 16 windows, drawn with a generator of your own (`torch.Generator().manual_seed(0)`), so the run's own
random numbers stay untouched. How many next-character questions does it hold? Put the number in `answer`.

In [8]:
answer = None   # TODO: xs, ys = get_batch(train_ids, 16, 32, torch.Generator().manual_seed(0)); then ys.numel()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 16 * 32, '16 windows, 32 targets each: ys.numel()'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
xs, ys = get_batch(train_ids, 16, 32, torch.Generator().manual_seed(0))
print(tuple(xs.shape), tuple(ys.shape))
answer = ys.numel()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 16 * 32, '16 windows, 32 targets each: ys.numel()'
    print("✓ right:", answer)

(16, 32) (16, 32)
✓ right: 512


## 2 · One number: the loss

The whole batch goes through one forward pass at once. Out come probabilities `[64×32×65]`: one set of 65 for each of the
2,048 questions. At every position, take the probability of the right answer and minus its natural log; average all
2,048. That one number is the **loss**. Before any training the probabilities are nearly even, so the first loss is close
to ln 65 ≈ 4.17, the loss of an even guess.

In [10]:
with torch.no_grad():
    logits, _ = model(x)                                           # [64×32×65] a score for every character
    probs = torch.softmax(logits, dim=-1)                          # [64×32×65] probabilities
    p_right = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [64×32]    the right answer's probability
    L = loss_of(model, x, y)                                       # []         one number
print('probabilities', tuple(probs.shape), ' right answers', tuple(p_right.shape), ' loss', tuple(L.shape))
print('first window, its first 5 right answers:', [round(v, 4) for v in p_right[0, :5].tolist()], ' an even guess:', round(1 / 65, 4))
print('loss', round(L.item(), 4), ' model.py says', round(model(x, y)[1].item(), 4), ' ln 65 =', round(math.log(65), 4))
assert tuple(probs.shape) == (64, 32, 65) and tuple(p_right.shape) == (64, 32)
assert abs(L.item() - model(x, y)[1].item()) < 1e-5           # loss_of is model.py's loss, step by step
assert abs(L.item() - 4.17812) < 2e-4               # the recorded run's first loss
assert abs(L.item() - math.log(65)) < 0.01                    # within 0.01 of an even guess

probabilities (64, 32, 65)  right answers (64, 32)  loss ()
first window, its first 5 right answers: [0.0139, 0.0162, 0.0159, 0.014, 0.0152]  an even guess: 0.0154
loss 4.1781  model.py says 4.1781  ln 65 = 4.1744


### ✋ Your turn

Compute the loss by hand for the first window alone: minus the natural log of its 32 right-answer probabilities (`p_right[0]`),
averaged. Put it in `answer`. Is it close to ln 65 too?

In [11]:
answer = None   # TODO: use p_right[0], .log() and .mean(), then .item()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (-p_right[0].log().mean().item())) < 1e-6, 'minus the log of p_right[0], then the mean: -p_right[0].log().mean().item()'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = -p_right[0].log().mean().item()
print(round(answer, 4), 'against ln 65 =', round(math.log(65), 4))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (-p_right[0].log().mean().item())) < 1e-6, 'minus the log of p_right[0], then the mean: -p_right[0].log().mean().item()'
    print("✓ right:", answer)

4.1928 against ln 65 = 4.1744
✓ right: 4.1927571296691895


## 3 · One step

`loss.backward()` runs **backpropagation**: it works back from the loss through every part of the model and gives each
learned number a **gradient**, stored beside it in `.grad` with the same shape. A gradient says which way, and how strongly,
nudging that number would change the loss. Then an optimizer, **AdamW**, moves every number a small step against its gradient.
The step size is the **learning rate**. The recorded recipe (`config.json`): AdamW with PyTorch's defaults besides the learning
rate, which warms up from 0 to 0.003 over 200 steps, then falls along a cosine to 0.0003 at step 100,000.

In [13]:
opt = torch.optim.AdamW(model.parameters(), lr=0.003)        # the recorded recipe: PyTorch's AdamW, its defaults besides lr
print({k: opt.defaults[k] for k in ('lr', 'betas', 'eps', 'weight_decay')})
loss = model(x, y)[1]                                         # []  the first batch's loss
loss.backward()                                               # backpropagation: a gradient for every learned number
for name, p in list(model.named_parameters())[:4]:
    print(f"  {name:22s} {str(tuple(p.shape)):8s} its gradient {str(tuple(p.grad.shape)):8s} first 3: {[round(v, 5) for v in p.grad.flatten()[:3].tolist()]}")
n_grads = sum(p.grad.numel() for p in model.parameters())
print(n_grads, 'gradients, one per learned number')
g = model.head_bias.grad                                      # [65] the readout bias: one gradient per character
print('readout bias gradients: space', round(g[tok.stoi[' ']].item(), 4), ' e', round(g[tok.stoi['e']].item(), 4), ' z', round(g[tok.stoi['z']].item(), 4))
assert n_grads == count_params(model) == 9417

{'lr': 0.003, 'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0.01}
  head_bias              (65,)    its gradient (65,)    first 3: [-0.01975, -0.14435, 0.01405]
  tok_emb.weight         (65, 28) its gradient (65, 28) first 3: [0.00737, 0.00408, 0.01392]
  pos_emb.weight         (32, 28) its gradient (32, 28) first 3: [0.0005, -0.00317, 0.00083]
  blocks.0.ln1.weight    (28,)    its gradient (28,)    first 3: [-0.0001, -0.00012, 0.00103]
9417 gradients, one per learned number
readout bias gradients: space -0.1443  e -0.0741  z 0.0163


In [14]:
before = [p.detach().clone() for p in model.parameters()]
for group in opt.param_groups:
    group['lr'] = lr_at(1)                                    # the first step's learning rate: still warming up
opt.step()                                                    # every number takes a small step against its gradient
moved = torch.cat([(p.detach() - b).abs().flatten() for p, b in zip(model.parameters(), before)])   # [9417]
with torch.no_grad():
    after = model(x, y)[1].item()
print('learning rate', lr_at(1), '·', int((moved > 0).sum()), 'numbers moved · the largest move', f"{moved.max().item():.7f}")
print('loss on this batch: before', round(loss.item(), 5), ' after', round(after, 5))
assert moved.max().item() < 2 * lr_at(1) and after < loss.item()   # small steps, and the loss went down a little

learning rate 1.5e-05 · 9417 numbers moved · the largest move 0.0000152
loss on this batch: before 4.17812  after 4.17765


`train_step` (above, in the teaching code) does all of this in one call: set the learning rate, compute the loss, clear the old
gradients, backpropagate, step.

### ✋ Your turn

Which of the 65 characters got the most negative readout-bias gradient, so that a step against it raises its score the most?
Put the character in `answer` (`model.head_bias.grad.argmin()` gives its id; `tok.chars` turns an id into its character). Is it
the most common right answer in the batch?

In [15]:
answer = None   # TODO: tok.chars[int(model.head_bias.grad.argmin())]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[int(model.head_bias.grad.argmin())], 'the id of the smallest gradient is model.head_bias.grad.argmin(); tok.chars[...] names it'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [16]:
#@title Show a solution { display-mode: "form" }
answer = tok.chars[int(model.head_bias.grad.argmin())]
print(repr(answer), '· the most common right answer in the batch:', repr(tok.chars[int(y.flatten().mode().values)]))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[int(model.head_bias.grad.argmin())], 'the id of the smallest gradient is model.head_bias.grad.argmin(); tok.chars[...] names it'
    print("✓ right:", answer)

' ' · the most common right answer in the batch: ' '
✓ right:  


## 4 · The learning loop

Batch, loss, gradients, step, and again: this cycle is the **learning loop**. Start again from the recorded seed and run its
first 500 steps exactly as the recorded run did: the same starting numbers, the same batches, the same learning rates. Every 100
steps, check the model on held-back text: 100 fixed batches from the last 10%, the same windows every time (the run drew them
once, from its own generator seeded 2024), so two checks differ only because the model changed. The recorded run checked every
500 steps.

In [17]:
g = torch.Generator().manual_seed(2024)                            # the run's own generator for its checks
fixed_train = [get_batch(train_ids, 64, 32, g) for _ in range(100)]   # drawn first, as the run did
held_back = [get_batch(val_ids, 64, 32, g) for _ in range(100)]       # held-back text: 100 batches of [64×32]
print(len(held_back), 'held-back batches of', tuple(held_back[0][0].shape), '·', sum(y.numel() for _, y in held_back), 'answers')

100 held-back batches of (64, 32) · 204800 answers


In [18]:
torch.manual_seed(1337)                                       # start again from the recorded seed
model = Prompter(cfg)                                       # the same starting numbers
opt = torch.optim.AdamW(model.parameters(), lr=0.003)
checks = {}
for step in range(500 + 1):
    if step % 100 == 0:
        checks[step] = average_loss(model, held_back)
        print(f"step {step:3d} · learning rate {lr_at(step):.6f} · held-back loss {checks[step]:.4f}")
    if step == 500:
        break
    x, y = get_batch(train_ids, 64, 32)                     # the next batch, from the run's own random numbers
    train_step(model, opt, x, y, lr_at(step + 1))           # one step
train_500 = average_loss(model, fixed_train)
print('step 500 on the fixed training batches:', round(train_500, 4))
# the recorded run's log at steps 0 and 500 (replayed by training/prompter/replay.py, matched to 1e-4)
assert abs(checks[0] - 4.17804) < 1e-4 and abs(checks[500] - 2.30359) < 1e-4 and abs(train_500 - 2.29827) < 1e-4

step   0 · learning rate 0.000000 · held-back loss 4.1780


step 100 · learning rate 0.001500 · held-back loss 3.2038


step 200 · learning rate 0.003000 · held-back loss 2.6038


step 300 · learning rate 0.003000 · held-back loss 2.4547


step 400 · learning rate 0.003000 · held-back loss 2.3706


step 500 · learning rate 0.003000 · held-back loss 2.3036


step 500 on the fixed training batches: 2.2983


Those are the recorded run's own numbers: its log says 4.17804 at step 0 and 2.30359 at step 500 on held-back text,
and 2.29827 on the fixed training batches. The recorded run went on for 100,000 steps, in
1,280 seconds on a CPU, with a check every 500 steps; its numbers at the best check were kept. Most of the fall comes
early; the rest is slow polishing. The numbers change only here, in training: never while the model writes.

In [19]:
RECORDED = {0: 4.178, 500: 2.3036, 1000: 2.1243, 5000: 2.017, 50000: 1.9654, 98000: 1.9351, 100000: 1.936}   # prompter-training.json: held-back checks
for step, val in RECORDED.items():
    print(f"step {step:7,d} · held-back loss {val:.4f}" + ("   ← the best check" if step == 98000 else ""))
assert abs(checks[500] - RECORDED[500]) < 2e-4 and min(RECORDED.values()) == RECORDED[98000]

step       0 · held-back loss 4.1780
step     500 · held-back loss 2.3036
step   1,000 · held-back loss 2.1243
step   5,000 · held-back loss 2.0170
step  50,000 · held-back loss 1.9654
step  98,000 · held-back loss 1.9351   ← the best check
step 100,000 · held-back loss 1.9360


### ✋ Your turn

What learning rate did step 100 use, still warming up? And what will the last step, 100,000, use? Put both in `answer` as a
pair, `(lr at 100, lr at 100,000)`.

In [20]:
answer = None   # TODO: (lr_at(...), lr_at(...))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer[0] - 0.0015) < 1e-9 and abs(answer[1] - 0.0003) < 1e-9, 'lr_at(100) is halfway up the warm-up; lr_at(100_000) is the end of the cosine'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [21]:
#@title Show a solution { display-mode: "form" }
answer = (lr_at(100), lr_at(100000))
print(answer)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer[0] - 0.0015) < 1e-9 and abs(answer[1] - 0.0003) < 1e-9, 'lr_at(100) is halfway up the warm-up; lr_at(100_000) is the end of the cosine'
    print("✓ right:", answer)

(0.0015, 0.0003)
✓ right: (0.0015, 0.0003)


## 5 · What it wrote along the way

Let the model write 240 characters from a new line, drawing each next character by its probability, with the same seed
(42) for its random draws each time: at step 0, after the 500 steps above, and at the recorded run's best step,
98,000 (its numbers, `prompter.json`, are the ones the course uses everywhere). The code never changes; only the numbers do.

In [22]:
def write(m, n=240, seed=42, prompt='\n'):
    """n characters after the prompt, each drawn by its probability (a seeded draw), as the course recorded them."""
    out = m.generate(torch.tensor([tok.encode(prompt)]), n, generator=torch.Generator().manual_seed(seed))
    return tok.decode(out[0, len(prompt):])

torch.manual_seed(1337)
step0 = Prompter(cfg)                      # the same seed, the same starting numbers: step 0 again
at_0, at_500 = write(step0), write(model)
print('--- step 0', at_0, sep='\n')
print('--- step 500', at_500, sep='\n')
RECORDED_TEXT = {0: "v:hDYqLMfJ\nvdfCaN:$wP &IAW:cGdn.,pSzoWDnqCOnmc.G HVyye&zaXMENFd\nlvmhgUX!qdrdRWI$k-\ntC;oDG;uxgRl3AB?i\nuwF!SWTqQ'u\nceWAiA3 \nKmUIQj!:U&vaD'&$RLUzJhDctR!MM?bNIgk-rfng O?Wrz\nSFPvnHWXSGQc,u-bQLU.U&hOVz:,?bqcQW?E$cUdHPR ;tHuTXFOoHN;CNZWCg;mWlK,uFL",
                 500: "\nKADYOLIDUCES:\nWhy wo my chard''st toannd bre fe tho einar heed lamstre,\nPre crake thall thegrl heeit, thancit'u\nI teie t\nHoure.\nCUWAUD nof Ozlerct, howbough-chn Ineer denavn thhef\nEstheth.\nDUOVI:, bqushe bes ith hinge thoun;\nMenngllol ture",
                 98000: '\nAs loud save may: whom grows no, onoth, but court of in and go lamstre!\n\nKISS:\nWhat all thegnings?\n\nDUKE VINCENTIO:\nAway\nKING HE:\nIt hand the how by the your for not prenatnge, likes both.\n\nPOMEOK:\nThat?\n\nGLOUCETES:\nThe of his he like ward'}   # prompter-samples.json, replay.json
assert at_0 == RECORDED_TEXT[0] and at_500 == RECORDED_TEXT[500]

--- step 0
v:hDYqLMfJ
vdfCaN:$wP &IAW:cGdn.,pSzoWDnqCOnmc.G HVyye&zaXMENFd
lvmhgUX!qdrdRWI$k-
tC;oDG;uxgRl3AB?i
uwF!SWTqQ'u
ceWAiA3 
KmUIQj!:U&vaD'&$RLUzJhDctR!MM?bNIgk-rfng O?Wrz
SFPvnHWXSGQc,u-bQLU.U&hOVz:,?bqcQW?E$cUdHPR ;tHuTXFOoHN;CNZWCg;mWlK,uFL
--- step 500

KADYOLIDUCES:
Why wo my chard''st toannd bre fe tho einar heed lamstre,
Pre crake thall thegrl heeit, thancit'u
I teie t
Houre.
CUWAUD nof Ozlerct, howbough-chn Ineer denavn thhef
Estheth.
DUOVI:, bqushe bes ith hinge thoun;
Menngllol ture


In [23]:
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy()) for t in exp['weights']['tensors']}
best = Prompter(cfg)
best.load_state_dict(state)
best.eval()
at_best = write(best)
print(f"--- step {exp['checkpoint_step']:,}, the best check", at_best, sep='\n')
assert exp['checkpoint_step'] == 98000 and at_best == RECORDED_TEXT[98000]

--- step 98,000, the best check

As loud save may: whom grows no, onoth, but court of in and go lamstre!

KISS:
What all thegnings?

DUKE VINCENTIO:
Away
KING HE:
It hand the how by the your for not prenatnge, likes both.

POMEOK:
That?

GLOUCETES:
The of his he like ward


### ✋ Your turn

A play is made of speeches, each under its speaker's name, a line such as `ROMEO:`. Count the lines that end with a colon in
each of the three texts, and put the three counts in `answer`: `[step 0, step 500, best step]`.

In [24]:
answer = None   # TODO: for each text, sum(line.endswith(':') for line in text.split('\n'))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [sum(l.endswith(':') for l in s.split('\n')) for s in (at_0, at_500, at_best)], "split each text into lines, and count the ones whose last character is ':'"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [25]:
#@title Show a solution { display-mode: "form" }
answer = [sum(line.endswith(':') for line in s.split('\n')) for s in (at_0, at_500, at_best)]
print(answer)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [sum(l.endswith(':') for l in s.split('\n')) for s in (at_0, at_500, at_best)], "split each text into lines, and count the ones whose last character is ':'"
    print("✓ right:", answer)

[0, 1, 5]
✓ right: [0, 1, 5]


## 6 · The result

The honest test is text the model never trained on. Score the best step's numbers on the whole held-back text, every character
once (as the run did at its end), and on the whole training text. Then the two baselines: an even guess, ln 65, and counting
pairs (episode F1: how often each character followed each other one in the training text, plus 1 for every pair).

In [26]:
def every_window(ids, T=32, per=256):
    """The whole text as windows side by side, every character scored once, in batches of 256 windows."""
    n = len(range(0, len(ids) - T - 1, T))
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×32] each
    return list(zip(x.split(per), y.split(per)))

held_best, train_best = average_loss(best, every_window(val_ids)), average_loss(best, every_window(train_ids))
held_500 = average_loss(model, every_window(val_ids))
even = math.log(65)
pairs = torch.ones(65, 65, dtype=torch.float64)                                   # [65×65] 1 for every pair, then the counts
pairs.index_put_((train_ids[:-1], train_ids[1:]), torch.ones(len(train_ids) - 1, dtype=torch.float64), accumulate=True)
pairs = pairs / pairs.sum(1, keepdim=True)                                          # each line adds up to 1
counting = -pairs.log()[val_ids[:-1], val_ids[1:]].mean().item()
print(f"held back: an even guess {even:.4f} · counting pairs {counting:.4f} · our 500 steps {held_500:.4f} · the best step {held_best:.4f}")
print(f"the best step on its training text: {train_best:.4f}")
assert abs(held_best - 1.9317) < 2e-4 and abs(train_best - 1.7651) < 2e-4   # the recorded scores
assert abs(counting - 2.4819) < 2e-4 and abs(even - 4.1744) < 1e-4
assert held_best < counting < even and train_best < held_best

held back: an even guess 4.1744 · counting pairs 2.4819 · our 500 steps 2.3022 · the best step 1.9317
the best step on its training text: 1.7651


On held-back text the trained model scores 1.93; on its training text, 1.77. The gap is small, so it learned patterns that
carry over to new text, not just the text it read. It beats counting pairs, 2.48, and an even guess, 4.17.
That is our tiny transformer, trained: the closed box from F0, opened and explained.

Next, optionally: **F1**, the counting baseline, built by hand. Then the deep dives build every part again, by hand and in code.

### ✋ Your turn

A loss is easier to feel as a probability: e^(−loss) is the typical probability the model gives the right next character.
Put the best step's, `math.exp(-held_best)`, in `answer`, then compare it with an even guess's, 1/65.

In [27]:
answer = None   # TODO: math.exp(-held_best)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - math.exp(-held_best)) < 1e-6, 'e to the minus loss: math.exp(-held_best)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [28]:
#@title Show a solution { display-mode: "form" }
answer = math.exp(-held_best)
for name, l in (('an even guess', even), ('counting pairs', counting), ('the best step', held_best)):
    print(f"{name:15s} {math.exp(-l):.3f}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - math.exp(-held_best)) < 1e-6, 'e to the minus loss: math.exp(-held_best)'
    print("✓ right:", answer)

an even guess   0.015
counting pairs  0.084
the best step   0.145
✓ right: 0.14490216465382452
